# Insurance Premium Prediction

**Authors:**
1. Lim Ming Jun

---

**Notebook 01 — Load Data.** Frames the business problem, loads and documents every source
dataset, and consolidates them into the canonical dataset that the downstream notebooks consume.

# 1 Data Foundation

## 1.1 Business Understanding

### 1.1.1 Problem Statement

Health insurers must price a policy **before** knowing what the policyholder will actually cost them. Set the premium too low and the insurer absorbs losses on high-cost members, set it too high and competitively-priced rivals take the low-risk customers, leaving the insurer with an increasingly expensive risk pool. Accurate, individual-level cost estimation is therefore central to both profitability and fair pricing.

Traditionally this estimation relies on actuarial tables and manual underwriting rules, which are coarse-grained and slow to adapt. A supervised learning model can instead learn the relationship between a member's attributes and their realised medical expenses directly from historical data.

**Objective**

Given a member's demographic and lifestyle attributes (age, sex, BMI, number of dependents, smoking status and region), predict the **annual individual medical charges billed by health insurance** — a continuous target, making this a **regression** problem.

**Goals**

1. **Predict** individual medical charges as accurately as possible, evaluated with RMSE, MAE and R².
2. **Explain** which attributes drive cost and by how much, so the model can inform underwriting rather than acting as a black box.
3. **Operationalise** the model behind an API so premium estimates can be served on demand.

**Why it matters**

- **Pricing** — premiums grounded in expected cost rather than broad risk bands.
- **Risk assessment** — early identification of high-cost members for care management.
- **Transparency** — quantified cost drivers (e.g. the premium attributable to smoking) support both regulatory scrutiny and customer-facing explanations.

## 1.2 Setup

This notebook only loads and documents data, so it imports only what it needs. Visualisation
(`seaborn`, `plotly`, `sweetviz`) and explainability (`shap`) belong to the notebooks that use them.

`PROJECT_ROOT` is resolved by walking up to the directory containing `pyproject.toml`, so the
paths hold regardless of the kernel's working directory.

In [1]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_DIR = PROJECT_ROOT / "data"

print(f"Project root : {PROJECT_ROOT}")
print(f"Data folder  : {DATA_DIR}")

Project root : C:\Users\ILLEGEAR\OneDrive\Desktop\Personal Project\DS & ML\Insurance Premium Prediction
Data folder  : C:\Users\ILLEGEAR\OneDrive\Desktop\Personal Project\DS & ML\Insurance Premium Prediction\data


## 1.3 Data Understanding

Every source dataset is documented below in the same four steps, in the same order:

| Step | Purpose |
|------|---------|
| **Load** | Read the raw file and preview the first rows. |
| **Schema** | Where the data comes from, and what each column means. |
| **Structure** | Row count, dtypes and null counts. |
| **Notes** | Observations that affect cleaning, encoding and modelling. |

This project uses a single source, so there is one subsection. Additional datasets would be
added as `1.3.2`, `1.3.3`, and so on, each following the same four steps.

### 1.3.1 Insurance Dataset

**Load**

In [2]:
# Load the raw insurance dataset from the "data" folder
ins_df = pd.read_csv(DATA_DIR / "insurance.csv")
ins_df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


**Schema**

**Source:** [US Health Insurance Dataset](https://www.kaggle.com/datasets/teertha/ushealthinsurancedataset) (Kaggle, uploaded by *teertha*) — a widely-used benchmark dataset of US health insurance beneficiaries, originally published with *Machine Learning with R* by Brett Lantz.

**Structure:** A single flat table of **1,338 rows × 7 columns**. Each row is **one insurance beneficiary**. There are no keys and no foreign relationships, so the dataset needs no relational modelling.

**Target:** `charges` — the remaining 6 columns are the predictors.

| Column | Data Type | Role | Description |
|--------|-----------|------|-------------|
| `age` | int | Feature | Age of the primary beneficiary, in years (**18 – 64**). |
| `sex` | object | Feature | Gender of the policyholder (`female`, `male`). |
| `bmi` | float | Feature | Body mass index, body weight relative to height in kg/m² (**15.96 – 53.13**). |
| `children` | int | Feature | Number of dependents covered by the insurance plan (**0 – 5**). |
| `smoker` | object | Feature | Whether the beneficiary smokes (`yes`, `no`). |
| `region` | object | Feature | Residential area in the US (`northeast`, `northwest`, `southeast`, `southwest`). |
| `charges` | float | **Target** | Individual medical costs billed by health insurance, in USD (**\$1,121.87 – \$63,770.43**). |

**Structure**

`info()` confirms the row count, column names, dtypes and null counts in a single view.

In [3]:
ins_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


**Notes**

Points that are not obvious from the schema above and that shape later decisions:

- **`smoker` dominates.** Smokers average **\$32,050** against **\$8,434** for non-smokers, correlating **0.787** with `charges` — far ahead of `age` (0.299) and `bmi` (0.198).
- **`charges` is strongly right-skewed** (skew **+1.52**, mean **\$13,270** vs median **\$9,382**). A minority of high-cost beneficiaries pull the mean well above the median, which may warrant a log transform or a model robust to skew.
- **`bmi` runs high.** The mean of **30.66** sits in the *obese* category, against a healthy range of 18.5 – 24.9.
- **Encoding.** `sex` and `smoker` are binary and map cleanly to 0/1. `region` is nominal with no natural order, so it should be one-hot encoded — treating it as ordinal would be meaningless.
- **Scale.** `charges` spans roughly two orders of magnitude while `children` spans 0 – 5, so distance- and gradient-based models will need feature scaling.
- **Data quality.** No missing values in any column. **1** fully duplicated row, to be reviewed during cleaning.

## 1.4 Data Consolidation

### 1.4.1 Join Strategy

With several sources, this is where the join keys, cardinality and join order would be decided,
then validated by comparing row counts before and after and checking for unmatched keys.

**This project needs no join.** The single source documented in 1.3.1 is already a flat table of
1,338 × 7 with no keys and no foreign relationships, so consolidation reduces to a pass-through.

We still emit `merged_data.csv` so that every downstream notebook depends on one canonical,
curated input rather than reaching back into raw data. `insurance.csv` remains the immutable
raw source and is never written to.

### 1.4.2 Save the Canonical Dataset

In [4]:
# Write the canonical dataset consumed by the downstream notebooks
output_path = DATA_DIR / "merged_data.csv"
ins_df.to_csv(output_path, index=False)

print(f"Saved {output_path.name}: {ins_df.shape[0]} rows x {ins_df.shape[1]} columns")

Saved merged_data.csv: 1338 rows x 7 columns
